In [187]:
from itertools import groupby
import json
import os
import sys
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.append("..")

from eurollm_eval.dataloader import EUROLLM_LANGS, EUROLLM_LPS

# Results processing

In [188]:
# Fetch relevant languages
EUROLLM_LANGS += [f"{lp[:2]}-{lp[-2:]}" for lp in EUROLLM_LPS]

EU_LANGS = [
    "bg",
    "hr",
    "cs",
    "da",
    "nl",
    "en",
    "et",
    "fi",
    "fr",
    "de",
    "el",
    "hu",
    "ga",
    "it",
    "lv",
    "lt",
    "mt",
    "pl",
    "pt",
    "ro",
    "sk",
    "sl",
    "es",
    "sv",
]
EU_LANGS += [
    f"{src_lang}-{tgt_lang}"
    for src_lang in EU_LANGS
    for tgt_lang in EU_LANGS
    if src_lang != tgt_lang
]

NON_EU_LANGS = list(set(EUROLLM_LANGS) - set(EU_LANGS)) + ["en"]
NON_EU_LANGS += [
    f"{src_lang}-{tgt_lang}"
    for src_lang in NON_EU_LANGS
    for tgt_lang in NON_EU_LANGS
    if src_lang != tgt_lang
]

In [189]:
# Tasks
BASE_TASKS = [
    "hellaswag_loglik",
    "mmlu_loglik",
    "arc_challenge_loglik",
    "m_hellaswag_loglik",
    "m_mmlu_loglik",
    "m_arc_challenge_loglik",
]

INSTRUCT_TASKS = [
    "hellaswag",
    "mmlu",
    "mmlu_pro",
    "bbh",
    "arc_challenge",
    "gpqa_diamond",
    "gsm8k",
    "math_500",
    "humaneval",
    "ifeval",
    "m_hellaswag",
    "m_mmlu",
    "mmlu_prox",
    "m_arc_challenge",
    "mgsm",
    "flores",
    "wmt24pp",
    "wmt25",
]

NEAT_TASK_NAMES = {
    "hellaswag": "Hellaswag",
    "mmlu": "MMLU",
    "mmlu_pro": "MMLU-Pro",
    "bbh": "BBH",
    "arc_challenge": "ARC-C",
    "gpqa_diamond": "GPQA ◆",
    "gsm8k": "GSM8K",
    "math_500": "MATH-500",
    "humaneval": "HumanEval",
    "ifeval": "IFEval",
    "m_hellaswag": "M-Hellaswag",
    "m_mmlu": "MMMLU",
    "mmlu_prox": "MMLU-ProX",
    "m_arc_challenge": "M-ARC-C",
    "mgsm": "MGSM",
    "flores": "FLORES",
    "wmt24pp": "WMT24++",
    "wmt25": "WMT25",
}

In [190]:
# Models
BASE_MODELS = [
    "EuroLLM-1.7B",
    "EuroMoE-2.6B-A0.6B-Preview",
    "EuroLLM-9B",
    "EuroLLM-9B-2512",
    "EuroLLM-22B-2512",
    "Apertus-8B-2509",
    "Apertus-70B-2509",
    "Mistral-Small-3.1-24B-Base-2503",
    "Olmo-3-1025-7B",
    "Olmo-3-1125-32B",
    "Llama-3.1-8B",
    "Llama-3.1-70B",
    "gemma-3-12b-pt",
    "gemma-3-27b-pt",
    "Qwen3-14B-Base",
    "Qwen3-30B-A3B-Base",
]

INSTRUCT_MODELS = [
    "EuroLLM-1.7B-Instruct",
    "EuroMoE-2.6B-A0.6B-Instruct-2512",
    "EuroLLM-9B-Instruct",
    "EuroLLM-9B-Instruct-2512",
    "EuroLLM-22B-Instruct-Preview",
    "EuroLLM-22B-Instruct-2512",
    "Apertus-8B-Instruct-2509",
    "Apertus-70B-Instruct-2509",
    "Mistral-Small-3.2-24B-Instruct-2506",
    "Olmo-3-7B-Instruct",
    "Olmo-3.1-32B-Instruct",
    "Llama-3.1-8B-Instruct",
    "Llama-3.3-70B-Instruct",
    "gemma-3-12b-it",
    "gemma-3-27b-it",
    "Qwen3-14B",
    "Qwen3-32B",
    "Qwen3-30B-A3B-Instruct-2507",
]

JUDGES = [
    "Llama-3_3-Nemotron-Super-49B-v1_5",
    "Qwen3-235B-A22B-Instruct-2507",
    "gpt-oss-120b",
]

MT_JUDGES = [
    "wmt22-comet-da",
]

NEAT_MODEL_NAMES = {
    "EuroLLM-1.7B-Instruct": "EuroLLM-1.7B",
    "EuroMoE-2.6B-A0.6B-Instruct-2512": "EuroMoE-2.6B-A0.6B",
    "EuroLLM-9B-Instruct": "EuroLLM-9B-Old",
    "EuroLLM-9B-Instruct-2512": "EuroLLM-9B",
    "EuroLLM-22B-Instruct-Preview": "EuroLLM-22B-Preview",
    "EuroLLM-22B-Instruct-2512": "EuroLLM-22B",    
    "Apertus-8B-Instruct-2509": "Apertus-8B",
    "Apertus-70B-Instruct-2509": "Apertus-70B",
    "Mistral-Small-3.2-24B-Instruct-2506": "Mistral-3.2-24B",
    "Olmo-3-7B-Instruct": "OLMo-3-7B",
    "Olmo-3.1-32B-Instruct": "OLMo-3.1-32B",
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B",
    "Llama-3.3-70B-Instruct": "Llama-3.3-70B",
    "gemma-3-12b-it": "Gemma-3-12B",
    "gemma-3-27b-it": "Gemma-3-27B",
    "Qwen3-14B": "Qwen-3-14B",
    "Qwen3-32B": "Qwen-3-32B",
    "Qwen3-30B-A3B-Instruct-2507": "Qwen-3-30B-A3B",
    "EuroLLM-1.7B": "EuroLLM-1.7B-Base",
    "EuroMoE-2.6B-A0.6B-Preview": "EuroMoE-2.6B-A0.6B-Base",
    "EuroLLM-9B": "EuroLLM-9B-Old-Base",
    "EuroLLM-9B-2512": "EuroLLM-9B-Base",
    "EuroLLM-22B-2512": "EuroLLM-22B-Base",
    "Apertus-8B-2509": "Apertus-8B-Base",
    "Apertus-70B-2509": "Apertus-70B-Base",
    "Mistral-Small-3.1-24B-Base-2503": "Mistral-3.2-24B-Base",
    "Olmo-3-1025-7B": "OLMo-3-7B-Base",
    "Olmo-3-1125-32B": "OLMo-3-32B-Base",
    "Llama-3.1-8B": "Llama-3.1-8B-Base",
    "Llama-3.1-70B": "Llama-3.3-70B-Base",
    "gemma-3-12b-pt": "Gemma-3-12B-Base",
    "gemma-3-27b-pt": "Gemma-3-27B-Base",
    "Qwen3-14B-Base": "Qwen-3-14B-Base",
    "Qwen3-30B-A3B-Base": "Qwen-3-30B-A3B-Base",
    "gpt-oss-120b": "GPT-OSS-120B",
    "Llama-3_3-Nemotron-Super-49B-v1_5": "Nemotron-1.5-49B",
    "Qwen3-235B-A22B-Instruct-2507": "Qwen-3-235B-A22B",
    "wmt22-comet-da": "COMET-22",
}

In [191]:
# Processing function
def fetch_results(results_path, output_path):
    results = {}

    for task in tqdm(BASE_TASKS if results_path.endswith("/base") else INSTRUCT_TASKS):
        neat_task_name = NEAT_TASK_NAMES[task.replace("_loglik", "")]
        results[neat_task_name] = {}
        
        if any(f"{task}_{lang}" in os.listdir(results_path) for lang in EUROLLM_LANGS):
            is_multilingual, is_mt = True, False
            tasks = [
                f"{task}_{lang}" for lang in EUROLLM_LANGS 
                if os.path.exists(f"{results_path}/{task}_{lang}")
            ]
            judges = JUDGES
        
        elif any(f"{task}_{lp}" in os.listdir(results_path) for lp in EUROLLM_LPS):
            is_multilingual, is_mt = True, True
            tasks = [
                f"{task}_{lp}" for lp in EUROLLM_LPS 
                if os.path.exists(f"{results_path}/{task}_{lp}")
            ]
            judges = MT_JUDGES
       
        else:
            is_multilingual, is_mt = False, False
            tasks = [task]
            judges = JUDGES

        for _task in tasks:            
            if is_multilingual:
                lang = _task.split("_")[-1]
                if is_mt:
                    lang = f"{lang[:2]}-{lang[-2:]}"
            else:
                lang = "en"

            results[neat_task_name][lang] = {}

            for model in (
                BASE_MODELS if results_path.endswith("/base") else INSTRUCT_MODELS
            ):
                if (
                    (model.split("-")[0] in ["EuroLLM", "EuroMoE"]) and
                    (task in ["flores", "wmt24pp", "wmt25"])
                ):
                    __task = _task.replace(task, f"{task}_eurollm")
                else:
                    __task = _task
                    
                neat_model_name = NEAT_MODEL_NAMES[model]
                results[neat_task_name][lang][neat_model_name] = {}

                if results_path.endswith("/base"):
                    if os.path.exists(f"{results_path}/{__task}/{model}/results.json"):
                        with open(
                            f"{results_path}/{__task}/{model}/results.json", "r"
                        ) as f:
                            results[neat_task_name][lang][neat_model_name] = (
                                json.load(f)["score"] * 100
                            )
                    else:
                        print(
                            f"No evaluation file found at {results_path}/{__task}/{model}"
                        )
                        results[neat_task_name][lang][neat_model_name] = None

                else:
                    for judge in judges:
                        neat_judge_name = NEAT_MODEL_NAMES[judge]

                        if os.path.exists(
                            f"{results_path}/{__task}/{model}/{judge}/results.json"
                        ):
                            with open(
                                f"{results_path}/{__task}/{model}/{judge}/results.json",
                                "r",
                            ) as f:
                                results[neat_task_name][lang][neat_model_name][
                                    neat_judge_name
                                ] = (json.load(f)["score"] * 100)
                        else:
                            print(
                                f"No evaluation file found at {results_path}/{__task}/{model}/{judge}"
                            )
                            results[neat_task_name][lang][neat_model_name][
                                neat_judge_name
                            ] = None

    os.makedirs(output_path, exist_ok=True)
    with open(f"{output_path}/results_{results_path.split('/')[-1]}.json", "w") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

In [192]:
# Fetch base results
fetch_results(
    results_path="../results/base",
    output_path="./",
)

100%|██████████| 6/6 [00:00<00:00,  8.18it/s]


In [193]:
# Fetch instruct results
fetch_results(
    results_path="../results/instruct",
    output_path="./",
)

100%|██████████| 18/18 [00:14<00:00,  1.28it/s]

No evaluation file found at ../results/instruct/wmt25_eurollm_csde/EuroLLM-9B-Instruct/wmt22-comet-da
No evaluation file found at ../results/instruct/wmt25_eurollm_csuk/EuroLLM-9B-Instruct/wmt22-comet-da
No evaluation file found at ../results/instruct/wmt25_eurollm_encs/EuroLLM-1.7B-Instruct/wmt22-comet-da
No evaluation file found at ../results/instruct/wmt25_eurollm_encs/EuroLLM-9B-Instruct/wmt22-comet-da
No evaluation file found at ../results/instruct/wmt25_eurollm_encs/EuroLLM-22B-Instruct-Preview/wmt22-comet-da
No evaluation file found at ../results/instruct/wmt25_eurollm_enet/EuroLLM-1.7B-Instruct/wmt22-comet-da
No evaluation file found at ../results/instruct/wmt25_eurollm_enet/EuroLLM-9B-Instruct/wmt22-comet-da
No evaluation file found at ../results/instruct/wmt25_eurollm_enet/EuroLLM-22B-Instruct-Preview/wmt22-comet-da
No evaluation file found at ../results/instruct/wmt25_eurollm_enar/EuroLLM-1.7B-Instruct/wmt22-comet-da
No evaluation file found at ../results/instruct/wmt25_euro

# Base models

In [194]:
# Load base results
with open("results_base.json", "r") as f:
    results = json.load(f)

In [195]:
# Define tasks
GROUPED_TASKS = {
    "English": [
        "Hellaswag",
        "MMLU",
        "ARC-C",
    ],
    "Multilingual": [
        "M-Hellaswag",
        "MMMLU",
        "M-ARC-C",
    ],
}

## EuroLLM-22B

In [196]:
# Define models
GROUPED_MODELS = {
    "Open-source": {
        "European": [
            "EuroLLM-9B-Base",
            "EuroLLM-22B-Base",
            "Apertus-8B-Base",
            "Apertus-70B-Base",
        ],
        "Non-European": [
            "OLMo-3-7B-Base",
            "OLMo-3-32B-Base",
        ],
    },
    "Open-weights": {
        "European": [
            "Mistral-3.2-24B-Base",
        ],
        "Non-European": [
            "Llama-3.1-8B-Base",
            "Llama-3.3-70B-Base",
            "Gemma-3-12B-Base",
            "Gemma-3-27B-Base",
            "Qwen-3-14B-Base",
            "Qwen-3-30B-A3B-Base",
        ],
    },
}

In [197]:
# Format results as dataframe
df = pd.DataFrame()

for task_group_1 in GROUPED_TASKS:
    is_multilingual = task_group_1 == "Multilingual"

    for task in GROUPED_TASKS[task_group_1]:
        if is_multilingual:
            langs = sorted(results[task].keys())
            for lang in langs:
                for model_group_1 in GROUPED_MODELS:
                    for model_group_2 in GROUPED_MODELS[model_group_1]:
                        for model in GROUPED_MODELS[model_group_1][model_group_2]:
                            try:
                                df.loc[
                                    f"{model_group_1}_{model_group_2}_{model}",
                                    f"{task_group_1}_{task}_{lang}",
                                ] = results[task][lang][model]
                            except:
                                df.loc[
                                    f"{model_group_1}_{model_group_2}_{model}",
                                    f"{task_group_1}_{task}_{lang}",
                                ] = None
        else:
            for model_group_1 in GROUPED_MODELS:
                for model_group_2 in GROUPED_MODELS[model_group_1]:
                    for model in GROUPED_MODELS[model_group_1][model_group_2]:
                        try:
                            df.loc[
                                f"{model_group_1}_{model_group_2}_{model}",
                                f"{task_group_1}_{task}_en",
                            ] = results[task]["en"][model]
                        except:
                            df.loc[
                                f"{model_group_1}_{model_group_2}_{model}",
                                f"{task_group_1}_{task}_en",
                            ] = None

df.index = pd.MultiIndex.from_tuples([idx.split("_") for idx in df.index])
df.columns = pd.MultiIndex.from_tuples([col.split("_") for col in df.columns])
df = df.fillna(np.nan)

In [198]:
# Aggregate results
df_agg = df.groupby(level=[0, 1], axis=1).mean()
df_agg = df_agg[
    [(group, task) for group, tasks in GROUPED_TASKS.items() for task in tasks]
]

df_en = df_agg["English"]

df_xx = df_agg["Multilingual"]

df_eu = df["Multilingual"].copy()
cols = df_eu.columns
for col in cols:
    if col[-1] not in EU_LANGS:
        df_eu = df_eu.drop(col, axis=1)
df_eu = df_eu.groupby(level=0, axis=1).mean()[GROUPED_TASKS["Multilingual"]]

df_non_eu = df["Multilingual"].copy()
cols = df_non_eu.columns
for col in cols:
    if col[-1] not in NON_EU_LANGS:
        df_non_eu = df_non_eu.drop(col, axis=1)
df_non_eu = df_non_eu.groupby(level=0, axis=1).mean()[GROUPED_TASKS["Multilingual"]]

In [199]:
# English results
df_en

Hellaswag       MMLU  \
Open-source  European     EuroLLM-9B-Base       72.238271  44.020229   
                          EuroLLM-22B-Base      73.164658  46.406439   
                          Apertus-8B-Base       73.224425  47.083126   
                          Apertus-70B-Base      77.408108  49.405228   
             Non-European OLMo-3-7B-Base        69.160275  45.957689   
                          OLMo-3-32B-Base       77.248730  52.005129   
Open-weights European     Mistral-3.2-24B-Base  79.310688  53.878481   
             Non-European Llama-3.1-8B-Base     75.714713  46.349455   
                          Llama-3.3-70B-Base    83.852973  54.939811   
                          Gemma-3-12B-Base      77.696982  51.926775   
                          Gemma-3-27B-Base      78.214962  54.654890   
                          Qwen-3-14B-Base       76.242654  54.248878   
                          Qwen-3-30B-A3B-Base   76.451838  52.973859   

                                                    ARC-C  
Open-source  European     EuroLLM-9B-Base       59.024808  
                          EuroLLM-22B-Base      62.275449  
                          Apertus-8B-Base       63.130881  
                          Apertus-70B-Base      63.986313  
             Non-European OLMo-3-7B-Base        61.762190  
                          OLMo-3-32B-Base       67.921300  
Open-weights European     Mistral-3.2-24B-Base  68.520103  
             Non-European Llama-3.1-8B-Base     58.254919  
                          Llama-3.3-70B-Base    68.434559  
                          Gemma-3-12B-Base      68.177930  
                          Gemma-3-27B-Base      70.487596  
                          Qwen-3-14B-Base       68.263473  
                          Qwen-3-30B-A3B-Base   60.564585

In [200]:
# Multilingual results
df_xx

M-Hellaswag      MMMLU  \
Open-source  European     EuroLLM-9B-Base         60.625118  39.103910   
                          EuroLLM-22B-Base        62.932884  41.501611   
                          Apertus-8B-Base         63.567680  40.855821   
                          Apertus-70B-Base        67.607230  42.286593   
             Non-European OLMo-3-7B-Base          40.834256  32.648793   
                          OLMo-3-32B-Base         54.164380  39.257409   
Open-weights European     Mistral-3.2-24B-Base    66.361134  46.146062   
             Non-European Llama-3.1-8B-Base       56.385250  36.977835   
                          Llama-3.3-70B-Base      69.343740  46.601990   
                          Gemma-3-12B-Base        65.731050  44.629570   
                          Gemma-3-27B-Base        68.877805  48.293826   
                          Qwen-3-14B-Base         61.196092  41.246285   
                          Qwen-3-30B-A3B-Base     62.629823  40.887197   

                                                  M-ARC-C  
Open-source  European     EuroLLM-9B-Base       50.308668  
                          EuroLLM-22B-Base      53.087729  
                          Apertus-8B-Base       52.596550  
                          Apertus-70B-Base      54.450929  
             Non-European OLMo-3-7B-Base        34.010365  
                          OLMo-3-32B-Base       46.894456  
Open-weights European     Mistral-3.2-24B-Base  57.987888  
             Non-European Llama-3.1-8B-Base     43.802291  
                          Llama-3.3-70B-Base    57.745914  
                          Gemma-3-12B-Base      57.878615  
                          Gemma-3-27B-Base      60.638028  
                          Qwen-3-14B-Base       54.297614  
                          Qwen-3-30B-A3B-Base   54.460499

In [201]:
# Results on European languages
df_eu

M-Hellaswag      MMMLU  \
Open-source  European     EuroLLM-9B-Base         62.518794  40.226008   
                          EuroLLM-22B-Base        64.772492  42.547895   
                          Apertus-8B-Base         65.520188  41.913945   
                          Apertus-70B-Base        69.653500  43.585616   
             Non-European OLMo-3-7B-Base          42.021096  33.577223   
                          OLMo-3-32B-Base         55.871483  40.523413   
Open-weights European     Mistral-3.2-24B-Base    68.634728  47.526089   
             Non-European Llama-3.1-8B-Base       57.918416  37.964921   
                          Llama-3.3-70B-Base      71.055893  47.769705   
                          Gemma-3-12B-Base        67.601952  45.708192   
                          Gemma-3-27B-Base        70.774745  49.457431   
                          Qwen-3-14B-Base         62.726043  45.119947   
                          Qwen-3-30B-A3B-Base     64.300769  44.832627   

                                                  M-ARC-C  
Open-source  European     EuroLLM-9B-Base       52.004680  
                          EuroLLM-22B-Base      54.886551  
                          Apertus-8B-Base       54.427944  
                          Apertus-70B-Base      56.709275  
             Non-European OLMo-3-7B-Base        35.071299  
                          OLMo-3-32B-Base       48.577949  
Open-weights European     Mistral-3.2-24B-Base  60.370017  
             Non-European Llama-3.1-8B-Base     44.997045  
                          Llama-3.3-70B-Base    59.313024  
                          Gemma-3-12B-Base      59.655556  
                          Gemma-3-27B-Base      62.672893  
                          Qwen-3-14B-Base       55.814629  
                          Qwen-3-30B-A3B-Base   55.887056

In [202]:
# Results on non_European languages
df_non_eu

M-Hellaswag      MMMLU  \
Open-source  European     EuroLLM-9B-Base         56.080294  36.859714   
                          EuroLLM-22B-Base        58.517825  39.409044   
                          Apertus-8B-Base         58.881660  38.739575   
                          Apertus-70B-Base        62.696182  39.688548   
             Non-European OLMo-3-7B-Base          37.985840  30.791932   
                          OLMo-3-32B-Base         50.067331  36.725403   
Open-weights European     Mistral-3.2-24B-Base    60.904508  43.386007   
             Non-European Llama-3.1-8B-Base       52.705653  35.003664   
                          Llama-3.3-70B-Base      65.234573  44.266560   
                          Gemma-3-12B-Base        61.240886  42.472328   
                          Gemma-3-27B-Base        64.325149  45.966616   
                          Qwen-3-14B-Base         57.524209  33.498960   
                          Qwen-3-30B-A3B-Base     58.619552  32.996337   

                                                  M-ARC-C  
Open-source  European     EuroLLM-9B-Base       46.916644  
                          EuroLLM-22B-Base      49.490084  
                          Apertus-8B-Base       48.933764  
                          Apertus-70B-Base      49.934238  
             Non-European OLMo-3-7B-Base        31.888496  
                          OLMo-3-32B-Base       43.527470  
Open-weights European     Mistral-3.2-24B-Base  53.223629  
             Non-European Llama-3.1-8B-Base     41.412783  
                          Llama-3.3-70B-Base    54.611694  
                          Gemma-3-12B-Base      54.324735  
                          Gemma-3-27B-Base      56.568299  
                          Qwen-3-14B-Base       51.263583  
                          Qwen-3-30B-A3B-Base   51.607385

In [203]:
# Detailed results on M-Hellaswag
df_task = df[("Multilingual", "M-Hellaswag")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU             \
                                                       da         de   
Open-source  European     EuroLLM-9B-Base       64.179746  63.352910   
                          EuroLLM-22B-Base      68.479897  64.452750   
                          Apertus-8B-Base       68.157386  65.904965   
                          Apertus-70B-Base      72.457536  70.678057   
             Non-European OLMo-3-7B-Base        39.378628  45.146823   
                          OLMo-3-32B-Base       55.740701  59.754405   
Open-weights European     Mistral-3.2-24B-Base  70.103204  71.158569   
             Non-European Llama-3.1-8B-Base     57.729521  59.295248   
                          Llama-3.3-70B-Base    73.586326  72.076882   
                          Gemma-3-12B-Base      71.264244  67.026161   
                          Gemma-3-27B-Base      73.554074  70.720769   
                          Qwen-3-14B-Base       61.922167  64.986652   
                          Qwen-3-30B-A3B-Base   63.846485  65.819541   

                                                                      \
                                                       es         fr   
Open-source  European     EuroLLM-9B-Base       66.097535  65.088377   
                          EuroLLM-22B-Base      67.591506  68.077129   
                          Apertus-8B-Base       69.064134  68.527049   
                          Apertus-70B-Base      73.023157  72.865560   
             Non-European OLMo-3-7B-Base        51.776758  51.622924   
                          OLMo-3-32B-Base       64.945043  65.024103   
Open-weights European     Mistral-3.2-24B-Base  73.908868  73.893948   
             Non-European Llama-3.1-8B-Base     64.304770  63.438672   
                          Llama-3.3-70B-Base    74.890620  73.829673   
                          Gemma-3-12B-Base      70.440721  70.208891   
                          Gemma-3-27B-Base      73.663430  73.797536   
                          Qwen-3-14B-Base       68.327820  68.248527   
                          Qwen-3-30B-A3B-Base   69.885818  69.416176   

                                                                      \
                                                       hr         hu   
Open-source  European     EuroLLM-9B-Base       58.998733  53.982010   
                          EuroLLM-22B-Base      62.452471  55.802984   
                          Apertus-8B-Base       63.350232  56.055287   
                          Apertus-70B-Base      67.279256  60.201843   
             Non-European OLMo-3-7B-Base        34.727503  31.570864   
                          OLMo-3-32B-Base       48.331221  39.282580   
Open-weights European     Mistral-3.2-24B-Base  64.607098  55.177710   
             Non-European Llama-3.1-8B-Base     51.901141  48.705573   
                          Llama-3.3-70B-Base    68.134770  61.627907   
                          Gemma-3-12B-Base      66.075201  58.150505   
                          Gemma-3-27B-Base      69.391635  61.748574   
                          Qwen-3-14B-Base       57.541191  52.885037   
                          Qwen-3-30B-A3B-Base   59.991550  54.464677   

                                                                      \
                                                       it         nl   
Open-source  European     EuroLLM-9B-Base       64.853101  65.309868   
                          EuroLLM-22B-Base      66.213275  66.821421   
                          Apertus-8B-Base       67.355822  67.566400   
                          Apertus-70B-Base      71.458107  71.885122   
             Non-European OLMo-3-7B-Base        45.495103  41.654070   
                          OLMo-3-32B-Base       61.349293  58.281149   
Open-weights European     Mistral-3.2-24B-Base  72.154516  70.276398   
             Non-European Llama-3.1-8B-Base     61.240479  60.915569   
                          Llama-3.3-70B-Base    73.079434  74.465558   
                    

In [204]:
# Detailed results on MMMLU
df_task = df[("Multilingual", "MMMLU")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU             \
                                                       da         de   
Open-source  European     EuroLLM-9B-Base       40.710445  40.920407   
                          EuroLLM-22B-Base      43.520412  43.017729   
                          Apertus-8B-Base       42.467621  43.576009   
                          Apertus-70B-Base      44.853442  45.084874   
             Non-European OLMo-3-7B-Base        33.174279  35.413052   
                          OLMo-3-32B-Base       40.490798  42.821577   
Open-weights European     Mistral-3.2-24B-Base  47.792168  49.294606   
             Non-European Llama-3.1-8B-Base     38.256457  39.170124   
                          Llama-3.3-70B-Base    47.739150  48.479819   
                          Gemma-3-12B-Base      46.527304  46.374953   
                          Gemma-3-27B-Base      50.632432  49.656733   
                          Qwen-3-14B-Base       44.785276  46.344776   
                          Qwen-3-30B-A3B-Base   45.277588  46.684270   

                                                                      \
                                                       es         fr   
Open-source  European     EuroLLM-9B-Base       41.804816  41.732885   
                          EuroLLM-22B-Base      43.987698  44.399450   
                          Apertus-8B-Base       43.725152  43.482579   
                          Apertus-70B-Base      45.555472  45.423289   
             Non-European OLMo-3-7B-Base        35.998800  36.842910   
                          OLMo-3-32B-Base       43.732653  43.711797   
Open-weights European     Mistral-3.2-24B-Base  49.193609  50.175733   
             Non-European Llama-3.1-8B-Base     39.861976  39.845660   
                          Llama-3.3-70B-Base    49.441152  50.863386   
                          Gemma-3-12B-Base      46.943215  47.066015   
                          Gemma-3-27B-Base      50.513840  51.405868   
                          Qwen-3-14B-Base       47.070737  48.471883   
                          Qwen-3-30B-A3B-Base   46.733178  47.264670   

                                                                      \
                                                       hr         hu   
Open-source  European     EuroLLM-9B-Base       38.699125  37.224484   
                          EuroLLM-22B-Base      40.639026  39.298057   
                          Apertus-8B-Base       40.646634  38.107672   
                          Apertus-70B-Base      41.993153  39.874050   
             Non-European OLMo-3-7B-Base        30.414606  29.582981   
                          OLMo-3-32B-Base       37.352606  34.475079   
Open-weights European     Mistral-3.2-24B-Base  45.446938  42.577375   
             Non-European Llama-3.1-8B-Base     36.021301  35.527225   
                          Llama-3.3-70B-Base    45.484975  44.658628   
                          Gemma-3-12B-Base      44.769874  42.201060   
                          Gemma-3-27B-Base      48.680107  46.355887   
                          Qwen-3-14B-Base       42.837581  41.087474   
                          Qwen-3-30B-A3B-Base   40.631419  40.810998   

                                                                      \
                                                       it         nl   
Open-source  European     EuroLLM-9B-Base       41.393381  40.754516   
                          EuroLLM-22B-Base      43.554481  42.864734   
                          Apertus-8B-Base       42.972646  42.166388   
                          Apertus-70B-Base      44.778601  43.487172   
             Non-European OLMo-3-7B-Base        34.887411  34.590861   
                          OLMo-3-32B-Base       42.836633  41.566722   
Open-weights European     Mistral-3.2-24B-Base  50.612060  47.252163   
             Non-European Llama-3.1-8B-Base     39.678102  38.894793   
                          Llama-3.3-70B-Base    50.264470  48.565356   
                    

In [205]:
# Detailed results on M-ARC-C
df_task = df[("Multilingual", "M-ARC-C")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU             \
                                                       da         de   
Open-source  European     EuroLLM-9B-Base       52.233677  53.430532   
                          EuroLLM-22B-Base      54.725086  57.032590   
                          Apertus-8B-Base       53.951890  56.689537   
                          Apertus-70B-Base      56.357388  58.233276   
             Non-European OLMo-3-7B-Base        32.216495  36.792453   
                          OLMo-3-32B-Base       46.563574  52.315609   
Open-weights European     Mistral-3.2-24B-Base  58.762887  63.979417   
             Non-European Llama-3.1-8B-Base     42.096220  47.598628   
                          Llama-3.3-70B-Base    56.099656  63.636364   
                          Gemma-3-12B-Base      59.020619  60.977702   
                          Gemma-3-27B-Base      61.769759  63.893654   
                          Qwen-3-14B-Base       52.749141  59.348199   
                          Qwen-3-30B-A3B-Base   56.013746  58.747856   

                                                                      \
                                                       es         fr   
Open-source  European     EuroLLM-9B-Base       56.041131  54.373928   
                          EuroLLM-22B-Base      57.240788  57.375643   
                          Apertus-8B-Base       57.926307  57.890223   
                          Apertus-70B-Base      60.925450  59.777015   
             Non-European OLMo-3-7B-Base        43.444730  42.795883   
                          OLMo-3-32B-Base       58.097686  54.888508   
Open-weights European     Mistral-3.2-24B-Base  65.209940  63.636364   
             Non-European Llama-3.1-8B-Base     50.728363  47.255575   
                          Llama-3.3-70B-Base    62.125107  61.749571   
                          Gemma-3-12B-Base      64.353042  60.806175   
                          Gemma-3-27B-Base      67.266495  63.550600   
                          Qwen-3-14B-Base       61.268209  57.975986   
                          Qwen-3-30B-A3B-Base   59.640103  58.147513   

                                                                      \
                                                       hr         hu   
Open-source  European     EuroLLM-9B-Base       45.111492  47.467811   
                          EuroLLM-22B-Base      49.656947  50.300429   
                          Apertus-8B-Base       51.457976  48.583691   
                          Apertus-70B-Base      54.030875  48.669528   
             Non-European OLMo-3-7B-Base        27.101201  27.982833   
                          OLMo-3-32B-Base       40.222985  36.051502   
Open-weights European     Mistral-3.2-24B-Base  55.831904  49.184549   
             Non-European Llama-3.1-8B-Base     41.938250  39.828326   
                          Llama-3.3-70B-Base    54.030875  54.077253   
                          Gemma-3-12B-Base      55.746141  52.532189   
                          Gemma-3-27B-Base      60.463122  55.536481   
                          Qwen-3-14B-Base       50.428816  49.785408   
                          Qwen-3-30B-A3B-Base   51.629503  51.845494   

                                                                      \
                                                       it         nl   
Open-source  European     EuroLLM-9B-Base       56.775300  52.915952   
                          EuroLLM-22B-Base      56.603774  56.689537   
                          Apertus-8B-Base       59.348199  55.746141   
                          Apertus-70B-Base      60.120069  57.718696   
             Non-European OLMo-3-7B-Base        37.993139  32.847341   
                          OLMo-3-32B-Base       54.974271  49.313894   
Open-weights European     Mistral-3.2-24B-Base  65.694683  60.463122   
             Non-European Llama-3.1-8B-Base     50.257290  43.653516   
                          Llama-3.3-70B-Base    62.521441  60.291595   
                    

## EuroMOE-2.6B-A0.6B

In [206]:
# Define models
MODELS = [
    "EuroLLM-1.7B-Base",
    "EuroMoE-2.6B-A0.6B-Base",
]

In [207]:
# Format results as dataframe
df = pd.DataFrame()

for task_group_1 in GROUPED_TASKS:
    is_multilingual = task_group_1 == "Multilingual"

    for task in GROUPED_TASKS[task_group_1]:
        if is_multilingual:
            langs = sorted(results[task].keys())
            for lang in langs:
                for model in MODELS:
                    try:
                        df.loc[model, f"{task_group_1}_{task}_{lang}"] = results[task][
                            lang
                        ][model]
                    except:
                        df.loc[model, f"{task_group_1}_{task}_{lang}"] = None
        else:
            for model in MODELS:
                try:
                    df.loc[model, f"{task_group_1}_{task}_en"] = results[task]["en"][
                        model
                    ]
                except:
                    df.loc[model, f"{task_group_1}_{task}_en"] = None

df.columns = pd.MultiIndex.from_tuples([col.split("_") for col in df.columns])
df = df.fillna(np.nan)

In [208]:
# Aggregate results
df_agg = df.groupby(level=[0, 1], axis=1).mean()
df_agg = df_agg[
    [(group, task) for group, tasks in GROUPED_TASKS.items() for task in tasks]
]

df_en = df_agg["English"]

df_xx = df_agg["Multilingual"]

df_eu = df["Multilingual"].copy()
cols = df_eu.columns
for col in cols:
    if col[-1] not in EU_LANGS:
        df_eu = df_eu.drop(col, axis=1)
df_eu = df_eu.groupby(level=0, axis=1).mean()[GROUPED_TASKS["Multilingual"]]

df_non_eu = df["Multilingual"].copy()
cols = df_non_eu.columns
for col in cols:
    if col[-1] not in NON_EU_LANGS:
        df_non_eu = df_non_eu.drop(col, axis=1)
df_non_eu = df_non_eu.groupby(level=0, axis=1).mean()[GROUPED_TASKS["Multilingual"]]

In [209]:
# English results
df_en

,Hellaswag,MMLU,ARC-C
EuroLLM-1.7B-Base,55.423847,34.055132,39.349872
EuroMoE-2.6B-A0.6B-Base,59.318657,36.548187,44.910180


In [210]:
# Multilingual results
df_xx

,M-Hellaswag,MMMLU,M-ARC-C
EuroLLM-1.7B-Base,44.167745,30.225034,31.923528
EuroMoE-2.6B-A0.6B-Base,47.461821,32.217901,35.899552


In [211]:
# Results on European languages
df_eu

,M-Hellaswag,MMMLU,M-ARC-C
EuroLLM-1.7B-Base,45.422002,30.818322,32.842322
EuroMoE-2.6B-A0.6B-Base,49.034620,33.061114,37.339778


In [212]:
# Results on non-European languages
df_non_eu

,M-Hellaswag,MMMLU,M-ARC-C
EuroLLM-1.7B-Base,41.157527,29.038457,30.085940
EuroMoE-2.6B-A0.6B-Base,43.687101,30.531476,33.019099


In [213]:
# Detailed results on M-Hellaswag
df_task = df[("Multilingual", "M-Hellaswag")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU                                   \
                                da         de         es         fr   
EuroLLM-1.7B-Base        45.936358  45.221570  49.653185  48.869845   
EuroMoE-2.6B-A0.6B-Base  50.139755  48.756006  53.377441  52.833423   

                                                                     \
                                hr         hu         it         nl   
EuroLLM-1.7B-Base        40.874525  39.271610  48.139282  47.538329   
EuroMoE-2.6B-A0.6B-Base  44.877482  41.443616  51.882481  50.971712   

                                                                     \
                                pt         ro         sk         sv   
EuroLLM-1.7B-Base        47.517884  43.540359  42.585952  45.915122   
EuroMoE-2.6B-A0.6B-Base  51.181444  46.970353  45.296351  50.685382   

                            Non-EU                                              
                                ar         ca         hi         ru         uk  
EuroLLM-1.7B-Base        39.867001  43.049522  36.353979  43.650879  42.866256  
EuroMoE-2.6B-A0.6B-Base  42.385261  46.644222  37.087007  46.369619  45.949394

In [214]:
# Detailed results on MMMLU
df_task = df[("Multilingual", "MMMLU")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU                                   \
                                da         de         es         fr   
EuroLLM-1.7B-Base        31.394380  31.814410  31.738054  31.593826   
EuroMoE-2.6B-A0.6B-Base  33.530258  33.504338  34.318506  34.306235   

                                                                     \
                                hr         hu         it         nl   
EuroLLM-1.7B-Base        29.174591  29.836418  31.215052  30.848641   
EuroMoE-2.6B-A0.6B-Base  31.213389  30.842485  34.358471  33.118263   

                                                                     \
                                pt         ro         sk         sv   
EuroLLM-1.7B-Base        31.461602  29.969026  29.956352  30.817515   
EuroMoE-2.6B-A0.6B-Base  33.578560  32.499811  31.817138  33.645920   

                            Non-EU                                   \
                                ar         ca         hi         ru   
EuroLLM-1.7B-Base        27.783366  30.266859  27.344378  29.660105   
EuroMoE-2.6B-A0.6B-Base  29.013540  32.471679  28.582918  31.444171   

                                               
                                uk         zh  
EuroLLM-1.7B-Base        28.808306  30.367725  
EuroMoE-2.6B-A0.6B-Base  30.807376  30.869169

In [215]:
# Detailed results on M-ARC-C
df_task = df[("Multilingual", "M-ARC-C")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU                                   \
                                da         de         es         fr   
EuroLLM-1.7B-Base        32.474227  32.504288  34.790060  33.533448   
EuroMoE-2.6B-A0.6B-Base  36.769759  37.821612  38.560411  38.850772   

                                                                     \
                                hr         hu         it         nl   
EuroLLM-1.7B-Base        30.102916  29.356223  32.504288  34.734134   
EuroMoE-2.6B-A0.6B-Base  32.933105  33.304721  39.451115  38.679245   

                                                                     \
                                pt         ro         sk         sv   
EuroLLM-1.7B-Base        35.818338  35.395189  29.073756  33.820998   
EuroMoE-2.6B-A0.6B-Base  40.616967  38.487973  34.219554  38.382100   

                            Non-EU                                   \
                                ar         ca         hi         ru   
EuroLLM-1.7B-Base        29.073756  29.492691  26.180258  31.132075   
EuroMoE-2.6B-A0.6B-Base  30.102916  36.113500  27.639485  35.334477   

                                               
                                uk         zh  
EuroLLM-1.7B-Base        31.217839  33.419023  
EuroMoE-2.6B-A0.6B-Base  34.562607  34.361611

# Instruct models

In [216]:
# Load instruct results
with open("results_instruct.json", "r") as f:
    results = json.load(f)

results[r"GPQA $\blacklozenge$"] = results["GPQA ◆"]
del results["GPQA ◆"]

In [217]:
# Define tasks
GROUPED_TASKS = {
    "English": {
        "Instruction-following": [
            "IFEval",
        ],
        "General": [
            "Hellaswag",
            "MMLU",
            "MMLU-Pro",
            "BBH",
        ],
        "STEM": [
            "ARC-C",
            r"GPQA $\blacklozenge$",
            "GSM8K",
            "MATH-500",
            "HumanEval",
        ],
    },
    "Multilingual": {
        "General": [
            "M-Hellaswag",
            "MMMLU",
            "MMLU-ProX",
        ],
        "STEM": [
            "M-ARC-C",
            "MGSM",
        ],
        "Translation": [
            "FLORES",
            "WMT24++",
            "WMT25",
        ],
    },
}

In [218]:
# Define judges
JUDGES = [
    "Nemotron-1.5-49B",
    "Qwen-3-235B-A22B",
    "GPT-OSS-120B",
]

MT_JUDGES = [
    "COMET-22",
]

## EuroLLM-22B

In [219]:
# Define models
GROUPED_MODELS = {
    "Open-source": {
        "European": [
            "EuroLLM-9B",
            "EuroLLM-22B-Preview",
            "EuroLLM-22B",
            "Apertus-8B",
            "Apertus-70B",
        ],
        "Non-European": [
            "OLMo-3-7B",
            "OLMo-3.1-32B",
        ],
    },
    "Open-weights": {
        "European": [
            "Mistral-3.2-24B",
        ],
        "Non-European": [
            "Llama-3.1-8B",
            "Llama-3.3-70B",
            "Gemma-3-12B",
            "Gemma-3-27B",
            "Qwen-3-14B",
            "Qwen-3-32B",
            "Qwen-3-30B-A3B",
        ],
    },
}

In [220]:
# Format results as dataframe
df = pd.DataFrame()

for task_group_1 in GROUPED_TASKS:
    is_multilingual = task_group_1 == "Multilingual"

    for task_group_2 in GROUPED_TASKS[task_group_1]:
        is_mt = task_group_2 == "Translation"

        for task in GROUPED_TASKS[task_group_1][task_group_2]:
            if is_multilingual:
                if is_mt:
                    lps = results[task]
                    lps = (
                        sorted([lp for lp in lps if lp.startswith("en")])
                        + sorted([lp for lp in lps if lp.endswith("en")])
                        + sorted(
                            [
                                lp
                                for lp in lps
                                if not (lp.startswith("en") or lp.endswith("en"))
                            ]
                        )
                    )
                    for lp in lps:
                        for model_group_1 in GROUPED_MODELS:
                            for model_group_2 in GROUPED_MODELS[model_group_1]:
                                for model in GROUPED_MODELS[model_group_1][
                                    model_group_2
                                ]:
                                    try:
                                        df.loc[
                                            f"{model_group_1}_{model_group_2}_{model}",
                                            f"{task_group_1}_{task_group_2}_{task}_{lp}",
                                        ] = np.mean(
                                            [
                                                results[task][lp][model][judge]
                                                for judge in MT_JUDGES
                                            ]
                                        )
                                    except:
                                        df.loc[
                                            f"{model_group_1}_{model_group_2}_{model}",
                                            f"{task_group_1}_{task_group_2}_{task}_{lp}",
                                        ] = None
                else:
                    langs = sorted(results[task].keys())
                    for lang in langs:
                        for model_group_1 in GROUPED_MODELS:
                            for model_group_2 in GROUPED_MODELS[model_group_1]:
                                for model in GROUPED_MODELS[model_group_1][
                                    model_group_2
                                ]:
                                    try:
                                        df.loc[
                                            f"{model_group_1}_{model_group_2}_{model}",
                                            f"{task_group_1}_{task_group_2}_{task}_{lang}",
                                        ] = np.mean(
                                            [
                                                results[task][lang][model][judge]
                                                for judge in JUDGES
                                            ]
                                        )
                                    except:
                                        df.loc[
                                            f"{model_group_1}_{model_group_2}_{model}",
                                            f"{task_group_1}_{task_group_2}_{task}_{lang}",
                                        ] = None
            else:
                for model_group_1 in GROUPED_MODELS:
                    for model_group_2 in GROUPED_MODELS[model_group_1]:
                        for model in GROUPED_MODELS[model_group_1][model_group_2]:
                            try:
                                df.loc[
                                    f"{model_group_1}_{model_group_2}_{model}",
                                    f"{task_group_1}_{task_group_2}_{task}_en",
                                ] = np.nanmean(
                                    [
                                        results[task]["en"][model][judge]
                                        for judge in JUDGES
                                    ]
                                )
                            except:
                                df.loc[
                                    f"{model_group_1}_{model_group_2}_{model}",
                                    f"{task_group_1}_{task_group_2}_{task}_en",
                                ] = None

df.index = pd.MultiIndex.from_tuples([idx.split("_") for idx in df.index])
df.columns = pd.MultiIndex.from_tuples([col.split("_") for col in df.columns])
df = df.fillna(np.nan)

In [221]:
# Aggregate results
df.loc[
    ("Open-source", "European", "EuroLLM-22B-Preview"),
    ("Multilingual", "Translation", "WMT25"),
] = None
df_agg = df.groupby(level=[0, 1, 2], axis=1).mean()
df_agg = df_agg[
    [
        (col1, col2, col3)
        for col1 in GROUPED_TASKS
        for col2 in GROUPED_TASKS[col1]
        for col3 in GROUPED_TASKS[col1][col2]
    ]
]

df_en = df_agg["English"]

df_xx = df_agg["Multilingual"]

df_eu = df["Multilingual"].copy()
cols = df_eu.columns
for col in cols:
    if col[-1] not in EU_LANGS:
        df_eu = df_eu.drop(col, axis=1)
df_eu = df_eu.groupby(level=[0, 1], axis=1).mean()
df_eu = df_eu[
    [
        (group, task)
        for group, tasks in GROUPED_TASKS["Multilingual"].items()
        for task in tasks
        if (group, task) in df_eu.columns
    ]
]

df_non_eu = df["Multilingual"].copy()
cols = df_non_eu.columns
for col in cols:
    if col[-1] not in NON_EU_LANGS:
        df_non_eu = df_non_eu.drop(col, axis=1)
df_non_eu = df_non_eu.groupby(level=[0, 1], axis=1).mean()
df_non_eu = df_non_eu[
    [
        (group, task)
        for group, tasks in GROUPED_TASKS["Multilingual"].items()
        for task in tasks
        if (group, task) in df_non_eu.columns
    ]
]

In [222]:
# English results
df_en

Instruction-following  \
                                                             IFEval   
Open-source  European     EuroLLM-9B                      62.415280   
                          EuroLLM-22B-Preview             61.614295   
                          EuroLLM-22B                     67.221195   
                          Apertus-8B                      59.149723   
                          Apertus-70B                     61.244609   
             Non-European OLMo-3-7B                       75.477511   
                          OLMo-3.1-32B                    84.226741   
Open-weights European     Mistral-3.2-24B                 65.742452   
             Non-European Llama-3.1-8B                    63.832409   
                          Llama-3.3-70B                   82.809612   
                          Gemma-3-12B                     76.463339   
                          Gemma-3-27B                     80.714726   
                          Qwen-3-14B                      81.577326   
                          Qwen-3-32B                      81.885397   
                          Qwen-3-30B-A3B                  83.672212   

                                                 General             \
                                               Hellaswag       MMLU   
Open-source  European     EuroLLM-9B           52.957578  65.505863   
                          EuroLLM-22B-Preview  74.284671  65.308835   
                          EuroLLM-22B          69.707230  69.811993   
                          Apertus-8B           58.079400  57.280539   
                          Apertus-70B          74.636527  67.851208   
             Non-European OLMo-3-7B            42.843391  69.299245   
                          OLMo-3.1-32B         75.844785  80.076437   
Open-weights European     Mistral-3.2-24B      83.993892  77.318046   
             Non-European Llama-3.1-8B         43.985262  68.321227   
                          Llama-3.3-70B        86.320786  84.615202   
                          Gemma-3-12B          83.203877  76.143000   
                          Gemma-3-27B          84.451968  80.446755   
                          Qwen-3-14B           86.748988  81.241988   
                          Qwen-3-32B           87.419505  83.974268   
                          Qwen-3-30B-A3B       88.232756  84.954660   

                                                                     \
                                                MMLU-Pro        BBH   
Open-source  European     EuroLLM-9B           42.273382  45.773819   
                          EuroLLM-22B-Preview  43.029699  53.903650   
                          EuroLLM-22B          50.847739  55.332002   
                          Apertus-8B           32.748781  42.789126   
                          Apertus-70B          41.935395  56.115292   
             Non-European OLMo-3-7B            56.920434  75.451800   
                          OLMo-3.1-32B         66.517066  85.327395   
Open-weights European     Mistral-3.2-24B      67.445146  78.134439   
             Non-European Llama-3.1-8B         45.797318  57.564122   
                          Llama-3.3-70B        70.412234  82.317104   
                          Gemma-3-12B          59.881981  78.364819   
                          Gemma-3-27B          66.627881  82.163518   
                          Qwen-3-14B           71.104832  83.484360   
                          Qwen-3-32B           74.077460  83.684022   
                          Qwen-3-30B-A3B       76.667775  86.126043   

                                                    STEM                       \
                                                   ARC-C GPQA $\blacklozenge$   
Open-source  European     EuroLLM-9B           85.864619            21.043771   
                          EuroLLM-22B-Preview  85.608646            25.084175   
                          EuroLLM-22B          89.761092            26.767677   
                 

In [223]:
# Multilingual results
df_xx

General             \
                                              M-Hellaswag      MMMLU   
Open-source  European     EuroLLM-9B            49.115610  60.185981   
                          EuroLLM-22B-Preview   64.965834  59.669704   
                          EuroLLM-22B           62.292867  64.101381   
                          Apertus-8B            50.244605  52.967361   
                          Apertus-70B           67.374322  60.332840   
             Non-European OLMo-3-7B             30.086474  48.349662   
                          OLMo-3.1-32B          47.419004  66.478343   
Open-weights European     Mistral-3.2-24B       83.134967  74.799594   
             Non-European Llama-3.1-8B          37.386602  52.943476   
                          Llama-3.3-70B         73.406298  78.425046   
                          Gemma-3-12B           73.510286  69.031651   
                          Gemma-3-27B           75.601886  74.580565   
                          Qwen-3-14B            76.433180  74.685305   
                          Qwen-3-32B            79.569986  78.968592   
                          Qwen-3-30B-A3B        78.480718  79.541370   

                                                               STEM  \
                                               MMLU-ProX    M-ARC-C   
Open-source  European     EuroLLM-9B           37.744544  79.558078   
                          EuroLLM-22B-Preview  37.857730  78.718247   
                          EuroLLM-22B          45.331033  82.663431   
                          Apertus-8B           29.481572  69.878700   
                          Apertus-70B          36.495045  78.593747   
             Non-European OLMo-3-7B            41.768412  54.572456   
                          OLMo-3.1-32B         57.012866  78.992151   
Open-weights European     Mistral-3.2-24B      64.085057  89.222551   
             Non-European Llama-3.1-8B         33.386990  68.143259   
                          Llama-3.3-70B        65.682618  90.129235   
                          Gemma-3-12B          53.322480  87.168479   
                          Gemma-3-27B          60.212239  90.091269   
                          Qwen-3-14B           66.070568  89.963881   
                          Qwen-3-32B           70.140602  92.549128   
                          Qwen-3-30B-A3B       71.984377  92.332134   

                                                         Translation  \
                                                    MGSM      FLORES   
Open-source  European     EuroLLM-9B           67.333333   88.810405   
                          EuroLLM-22B-Preview  71.866667   88.851412   
                          EuroLLM-22B          76.066667   88.839978   
                          Apertus-8B           58.888889   87.768765   
                          Apertus-70B          72.733333   88.051950   
             Non-European OLMo-3-7B            76.644444   72.082763   
                          OLMo-3.1-32B         87.377778   81.960548   
Open-weights European     Mistral-3.2-24B      89.577778   86.987414   
             Non-European Llama-3.1-8B         73.000000   85.504705   
                          Llama-3.3-70B        91.644444   88.121640   
                          Gemma-3-12B          86.044444   88.154350   
                          Gemma-3-27B          88.444444   88.855239   
                          Qwen-3-14B           90.044444   86.131089   
                          Qwen-3-32B           91.688889   86.436439   
                          Qwen-3-30B-A3B       90.533333   86.732792   

                                                                     
                                                 WMT24++      WMT25  
Open-source  European     EuroLLM-9B           83.310434  80.229790  
                          EuroLLM-22B-Preview  83.583118        NaN  
                          EuroLLM-22B          83.521609  79.688340  
                          Apertus-8B      

In [224]:
# Results on European languages
df_eu

General             \
                                              M-Hellaswag      MMMLU   
Open-source  European     EuroLLM-9B            49.884691  61.450194   
                          EuroLLM-22B-Preview   66.444275  61.195429   
                          EuroLLM-22B           62.556022  65.597413   
                          Apertus-8B            50.888338  54.009494   
                          Apertus-70B           68.558030  61.690169   
             Non-European OLMo-3-7B             29.987447  49.331237   
                          OLMo-3.1-32B          49.153282  68.231529   
Open-weights European     Mistral-3.2-24B       84.268574  75.969526   
             Non-European Llama-3.1-8B          37.695818  54.311487   
                          Llama-3.3-70B         74.728399  79.866086   
                          Gemma-3-12B           74.513434  70.266840   
                          Gemma-3-27B           76.427002  75.814775   
                          Qwen-3-14B            77.539031  75.834647   
                          Qwen-3-32B            80.485191  79.936870   
                          Qwen-3-30B-A3B        79.269776  80.631830   

                                                               STEM  \
                                               MMLU-ProX    M-ARC-C   
Open-source  European     EuroLLM-9B           38.958609  80.749897   
                          EuroLLM-22B-Preview  39.349799  80.015647   
                          EuroLLM-22B          46.814395  84.078831   
                          Apertus-8B           30.409129  71.009523   
                          Apertus-70B          37.781800  79.575092   
             Non-European OLMo-3-7B            42.994829  54.521471   
                          OLMo-3.1-32B         58.921029  79.842147   
Open-weights European     Mistral-3.2-24B      65.634023  90.043086   
             Non-European Llama-3.1-8B         35.628637  69.003466   
                          Llama-3.3-70B        68.013558  91.122619   
                          Gemma-3-12B          54.866586  87.924971   
                          Gemma-3-27B          61.589704  90.763824   
                          Qwen-3-14B           67.499666  90.549028   
                          Qwen-3-32B           71.266588  93.050488   
                          Qwen-3-30B-A3B       73.099024  93.100630   

                                                         Translation  \
                                                    MGSM      FLORES   
Open-source  European     EuroLLM-9B           71.022222   88.870676   
                          EuroLLM-22B-Preview  73.866667   88.927236   
                          EuroLLM-22B          77.777778   88.892272   
                          Apertus-8B           61.377778   87.752781   
                          Apertus-70B          73.600000   88.083819   
             Non-European OLMo-3-7B            80.577778   69.192276   
                          OLMo-3.1-32B         88.800000   80.429699   
Open-weights European     Mistral-3.2-24B      90.755556   86.732157   
             Non-European Llama-3.1-8B         75.600000   85.133596   
                          Llama-3.3-70B        92.977778   88.056584   
                          Gemma-3-12B          87.511111   88.000915   
                          Gemma-3-27B          89.911111   88.789228   
                          Qwen-3-14B           90.311111   85.498463   
                          Qwen-3-32B           91.955556   85.876397   
                          Qwen-3-30B-A3B       91.377778   86.207159   

                                                                     
                                                 WMT24++      WMT25  
Open-source  European     EuroLLM-9B           83.607911  80.361044  
                          EuroLLM-22B-Preview  83.946212        NaN  
                          EuroLLM-22B          83.863356  79.905252  
                          Apertus-8B      

In [225]:
# Results on European languages
df_non_eu

General             \
                                              M-Hellaswag      MMMLU   
Open-source  European     EuroLLM-9B            47.269814  57.657556   
                          EuroLLM-22B-Preview   61.417576  56.618255   
                          EuroLLM-22B           61.661294  61.109318   
                          Apertus-8B            48.699647  50.883094   
                          Apertus-70B           64.533424  57.618181   
             Non-European OLMo-3-7B             30.324137  46.386511   
                          OLMo-3.1-32B          43.256736  62.971970   
Open-weights European     Mistral-3.2-24B       80.414311  72.459730   
             Non-European Llama-3.1-8B          36.644483  50.207454   
                          Llama-3.3-70B         70.233253  75.542968   
                          Gemma-3-12B           71.102730  66.561275   
                          Gemma-3-27B           73.621609  72.112145   
                          Qwen-3-14B            73.779139  72.386620   
                          Qwen-3-32B            77.373494  77.032035   
                          Qwen-3-30B-A3B        76.586976  77.360450   

                                                               STEM  \
                                               MMLU-ProX    M-ARC-C   
Open-source  European     EuroLLM-9B           36.530479  77.174440   
                          EuroLLM-22B-Preview  36.365661  76.123447   
                          EuroLLM-22B          43.847671  79.832630   
                          Apertus-8B           28.554015  67.617052   
                          Apertus-70B          35.208290  76.631058   
             Non-European OLMo-3-7B            40.541996  54.674427   
                          OLMo-3.1-32B         55.104702  77.292158   
Open-weights European     Mistral-3.2-24B      62.536092  87.581482   
             Non-European Llama-3.1-8B         31.145344  66.422845   
                          Llama-3.3-70B        63.351678  88.142468   
                          Gemma-3-12B          51.778374  85.655496   
                          Gemma-3-27B          58.834773  88.746158   
                          Qwen-3-14B           64.641470  88.793589   
                          Qwen-3-32B           69.014615  91.546410   
                          Qwen-3-30B-A3B       70.869729  90.795140   

                                                         Translation  \
                                                    MGSM      FLORES   
Open-source  European     EuroLLM-9B           63.644444   88.663074   
                          EuroLLM-22B-Preview  69.866667   88.666065   
                          EuroLLM-22B          74.355556   88.712149   
                          Apertus-8B           56.400000   87.807837   
                          Apertus-70B          71.866667   87.974050   
             Non-European OLMo-3-7B            72.711111   79.148397   
                          OLMo-3.1-32B         85.955556   85.702622   
Open-weights European     Mistral-3.2-24B      88.400000   87.611375   
             Non-European Llama-3.1-8B         70.400000   86.411860   
                          Llama-3.3-70B        90.311111   88.280664   
                          Gemma-3-12B          84.577778   88.529412   
                          Gemma-3-27B          86.977778   89.016601   
                          Qwen-3-14B           89.777778   87.677509   
                          Qwen-3-32B           91.422222   87.805430   
                          Qwen-3-30B-A3B       89.688889   88.017673   

                                                                     
                                                 WMT24++      WMT25  
Open-source  European     EuroLLM-9B           82.685733  80.180569  
                          EuroLLM-22B-Preview  82.820619        NaN  
                          EuroLLM-22B          82.803939  79.606998  
                          Apertus-8B      

In [226]:
# Detailed results on M-Hellaswag
df_task = df[("Multilingual", "General", "M-Hellaswag")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU             \
                                                      da         de   
Open-source  European     EuroLLM-9B           48.658427  47.487902   
                          EuroLLM-22B-Preview  68.181981  68.257188   
                          EuroLLM-22B          60.136128  64.279106   
                          Apertus-8B           51.366649  54.027896   
                          Apertus-70B          68.826796  70.562909   
             Non-European OLMo-3-7B            28.833960  35.126672   
                          OLMo-3.1-32B         49.905069  57.052377   
Open-weights European     Mistral-3.2-24B      85.172846  87.058782   
             Non-European Llama-3.1-8B         32.917786  42.047395   
                          Llama-3.3-70B        73.730969  75.295332   
                          Gemma-3-12B          76.055884  75.494592   
                          Gemma-3-27B          78.104961  77.647310   
                          Qwen-3-14B           77.689414  80.212781   
                          Qwen-3-32B           80.985133  82.468688   
                          Qwen-3-30B-A3B       79.168906  81.312269   

                                                                     \
                                                      es         fr   
Open-source  European     EuroLLM-9B           52.279354  53.062754   
                          EuroLLM-22B-Preview  68.476638  69.379596   
                          EuroLLM-22B          67.235616  65.556507   
                          Apertus-8B           53.986203  53.730278   
                          Apertus-70B          71.922338  69.854359   
             Non-European OLMo-3-7B            35.907830  34.771900   
                          OLMo-3.1-32B         57.567029  57.578354   
Open-weights European     Mistral-3.2-24B      87.280421  87.434854   
             Non-European Llama-3.1-8B         39.118839  35.849932   
                          Llama-3.3-70B        78.294574  77.793246   
                          Gemma-3-12B          76.605505  75.640751   
                          Gemma-3-27B          78.123889  77.189976   
                          Qwen-3-14B           81.626485  80.745342   
                          Qwen-3-32B           83.176872  83.115585   
                          Qwen-3-30B-A3B       82.739492  83.451132   

                                                                     \
                                                      hr         hu   
Open-source  European     EuroLLM-9B           48.037870  48.448295   
                          EuroLLM-22B-Preview  61.855489  60.584128   
                          EuroLLM-22B          59.148981  58.803962   
                          Apertus-8B           49.273220  44.346968   
                          Apertus-70B          66.568120  61.311547   
             Non-European OLMo-3-7B            23.401260  17.198523   
                          OLMo-3.1-32B         41.442298  32.441423   
Open-weights European     Mistral-3.2-24B      80.776405  75.965932   
             Non-European Llama-3.1-8B         34.639074  36.941185   
                          Llama-3.3-70B        70.200260  69.192528   
                          Gemma-3-12B          73.191849  67.814453   
                          Gemma-3-27B          75.145180  69.547099   
                          Qwen-3-14B           74.648928  69.086523   
                          Qwen-3-32B           77.281526  74.452608   
                          Qwen-3-30B-A3B       74.328652  72.778448   

                                                                     \
                                                      it         nl   
Open-source  European     EuroLLM-9B           49.860401  53.549199   
                          EuroLLM-22B-Preview  66.956742  68.659831   
                          EuroLLM-22B          64.523732  65.767224   
                          Apertus-8B           52.902571  53.621155   
    

In [227]:
# Detailed results on MMMLU
df_task = df[("Multilingual", "General", "MMMLU")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU             \
                                                      da         de   
Open-source  European     EuroLLM-9B           61.424100  62.005330   
                          EuroLLM-22B-Preview  61.355949  61.301353   
                          EuroLLM-22B          65.364228  66.369990   
                          Apertus-8B           54.078954  54.754362   
                          Apertus-70B          61.429148  62.224066   
             Non-European OLMo-3-7B            48.530971  52.974305   
                          OLMo-3.1-32B         67.812610  70.458088   
Open-weights European     Mistral-3.2-24B      76.349134  76.328757   
             Non-European Llama-3.1-8B         52.180827  57.127772   
                          Llama-3.3-70B        79.234186  80.864384   
                          Gemma-3-12B          70.667373  70.541057   
                          Gemma-3-27B          76.046242  75.582039   
                          Qwen-3-14B           75.460649  76.336300   
                          Qwen-3-32B           80.180726  79.946699   
                          Qwen-3-30B-A3B       80.281690  81.563333   

                                                                     \
                                                      es         fr   
Open-source  European     EuroLLM-9B           63.026849  62.984748   
                          EuroLLM-22B-Preview  62.386881  62.730120   
                          EuroLLM-22B          68.019099  67.636799   
                          Apertus-8B           55.989701  55.371375   
                          Apertus-70B          63.811809  63.315764   
             Non-European OLMo-3-7B            55.779711  56.158175   
                          OLMo-3.1-32B         72.286386  72.665699   
Open-weights European     Mistral-3.2-24B      78.451077  77.921218   
             Non-European Llama-3.1-8B         59.324534  59.335931   
                          Llama-3.3-70B        81.935903  81.282306   
                          Gemma-3-12B          71.448928  71.726122   
                          Gemma-3-27B          76.591170  76.907799   
                          Qwen-3-14B           77.903605  77.552008   
                          Qwen-3-32B           81.255937  81.261936   
                          Qwen-3-30B-A3B       82.100895  82.120032   

                                                                     \
                                                      hr         hu   
Open-source  European     EuroLLM-9B           58.774465  58.525287   
                          EuroLLM-22B-Preview  58.526012  58.699324   
                          EuroLLM-22B          62.716763  62.200553   
                          Apertus-8B           52.324815  51.699427   
                          Apertus-70B          60.232228  58.440827   
             Non-European OLMo-3-7B            43.291755  35.739148   
                          OLMo-3.1-32B         63.817057  59.147215   
Open-weights European     Mistral-3.2-24B      73.230403  71.068796   
             Non-European Llama-3.1-8B         46.863908  51.346233   
                          Llama-3.3-70B        77.403407  76.297604   
                          Gemma-3-12B          68.230910  66.873976   
                          Gemma-3-27B          74.168441  72.765663   
                          Qwen-3-14B           73.496603  72.079750   
                          Qwen-3-32B           78.300882  76.904177   
                          Qwen-3-30B-A3B       78.660886  77.364865   

                                                                     \
                                                      it         nl   
Open-source  European     EuroLLM-9B           62.418473  62.067744   
                          EuroLLM-22B-Preview  62.201909  62.105689   
                          EuroLLM-22B          66.855531  66.406112   
                          Apertus-8B           54.544081  53.767929   
    

In [228]:
# Detailed results on MMLU-ProX
df_task = df[("Multilingual", "General", "MMLU-ProX")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU             \
                                                      cs         de   
Open-source  European     EuroLLM-9B           39.093460  38.271395   
                          EuroLLM-22B-Preview  39.204014  38.815659   
                          EuroLLM-22B          46.775519  46.058338   
                          Apertus-8B           30.453270  30.600675   
                          Apertus-70B          37.540040  37.678941   
             Non-European OLMo-3-7B            38.625733  45.361000   
                          OLMo-3.1-32B         56.688494  59.511863   
Open-weights European     Mistral-3.2-24B      64.818437  65.844601   
             Non-European Llama-3.1-8B         33.381523  36.227570   
                          Llama-3.3-70B        67.845905  67.752360   
                          Gemma-3-12B          54.108909  54.534116   
                          Gemma-3-27B          61.116308  61.209853   
                          Qwen-3-14B           66.808402  67.086203   
                          Qwen-3-32B           71.026448  70.720299   
                          Qwen-3-30B-A3B       72.290728  72.837826   

                                                                     \
                                                      es         fr   
Open-source  European     EuroLLM-9B           39.473311  39.955212   
                          EuroLLM-22B-Preview  39.725600  40.077104   
                          EuroLLM-22B          47.472858  47.577742   
                          Apertus-8B           30.810443  30.257675   
                          Apertus-70B          38.342263  38.194858   
             Non-European OLMo-3-7B            47.674122  47.404825   
                          OLMo-3.1-32B         61.280721  61.377101   
Open-weights European     Mistral-3.2-24B      66.615642  66.595799   
             Non-European Llama-3.1-8B         38.214701  38.376279   
                          Llama-3.3-70B        68.670805  68.761516   
                          Gemma-3-12B          55.767214  55.322165   
                          Gemma-3-27B          62.085778  61.983729   
                          Qwen-3-14B           68.441194  67.440542   
                          Qwen-3-32B           72.106472  71.573547   
                          Qwen-3-30B-A3B       73.949032  73.827139   

                                                                     \
                                                      hu         it   
Open-source  European     EuroLLM-9B           37.508858  39.028262   
                          EuroLLM-22B-Preview  37.829180  40.162145   
                          EuroLLM-22B          45.423364  47.217734   
                          Apertus-8B           29.991212  30.155625   
                          Apertus-70B          36.973099  37.684610   
             Non-European OLMo-3-7B            30.291691  45.757859   
                          OLMo-3.1-32B         52.535646  60.359441   
Open-weights European     Mistral-3.2-24B      62.253026  66.493749   
             Non-European Llama-3.1-8B         30.376733  35.805199   
                          Llama-3.3-70B        66.312328  67.837401   
                          Gemma-3-12B          53.179125  55.531933   
                          Gemma-3-27B          60.039119  62.632877   
                          Qwen-3-14B           66.142246  68.157723   
                          Qwen-3-32B           69.773507  71.548034   
                          Qwen-3-30B-A3B       71.528191  73.798792   

                                                             Non-EU  \
                                                      pt         ar   
Open-source  European     EuroLLM-9B           39.379766  36.006463   
                          EuroLLM-22B-Preview  39.634890  36.335289   
                          EuroLLM-22B          47.175213  43.371035   
                          Apertus-8B           30.595005  28.148652   
    

In [229]:
# Detailed results on M-ARC-C
df_task = df[("Multilingual", "STEM", "M-ARC-C")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU             \
                                                      da         de   
Open-source  European     EuroLLM-9B           81.119680  81.380097   
                          EuroLLM-22B-Preview  80.605541  81.266039   
                          EuroLLM-22B          86.118252  84.573710   
                          Apertus-8B           70.437018  73.510123   
                          Apertus-70B          78.149100  78.414599   
             Non-European OLMo-3-7B            53.241931  60.222412   
                          OLMo-3.1-32B         78.777492  84.830339   
Open-weights European     Mistral-3.2-24B      90.117109  91.730824   
             Non-European Llama-3.1-8B         64.495858  73.880810   
                          Llama-3.3-70B        90.859754  91.616766   
                          Gemma-3-12B          87.603542  88.166524   
                          Gemma-3-27B          91.231077  91.189050   
                          Qwen-3-14B           89.917167  90.447676   
                          Qwen-3-32B           92.402171  93.584260   
                          Qwen-3-30B-A3B       92.259354  93.840890   

                                                                     \
                                                      es         fr   
Open-source  European     EuroLLM-9B           83.361823  80.895352   
                          EuroLLM-22B-Preview  82.051282  80.011406   
                          EuroLLM-22B          86.182336  83.832335   
                          Apertus-8B           72.649573  73.339036   
                          Apertus-70B          82.934473  81.779299   
             Non-European OLMo-3-7B            67.378917  67.379527   
                          OLMo-3.1-32B         86.780627  87.339607   
Open-weights European     Mistral-3.2-24B      91.481481  90.362133   
             Non-European Llama-3.1-8B         75.754986  75.220987   
                          Llama-3.3-70B        92.478632  92.044482   
                          Gemma-3-12B          88.774929  89.136014   
                          Gemma-3-27B          91.680912  91.246079   
                          Qwen-3-14B           92.849003  92.015968   
                          Qwen-3-32B           94.558405  93.498717   
                          Qwen-3-30B-A3B       94.216524  93.527231   

                                                                     \
                                                      hr         hu   
Open-source  European     EuroLLM-9B           76.190476  77.368721   
                          EuroLLM-22B-Preview  77.102937  75.342466   
                          EuroLLM-22B          82.149986  80.565068   
                          Apertus-8B           69.175934  67.636986   
                          Apertus-70B          77.844311  77.425799   
             Non-European OLMo-3-7B            42.087254  31.107306   
                          OLMo-3.1-32B         73.339036  64.954338   
Open-weights European     Mistral-3.2-24B      87.482179  86.558219   
             Non-European Llama-3.1-8B         59.566581  65.011416   
                          Llama-3.3-70B        89.193042  89.611872   
                          Gemma-3-12B          87.311092  84.389269   
                          Gemma-3-27B          89.078985  88.127854   
                          Qwen-3-14B           88.736812  87.300228   
                          Qwen-3-32B           91.445680  91.381279   
                          Qwen-3-30B-A3B       91.132022  91.238584   

                                                                     \
                                                      it         nl   
Open-source  European     EuroLLM-9B           83.262047  82.492159   
                          EuroLLM-22B-Preview  81.864842  81.123467   
                          EuroLLM-22B          85.771315  84.260051   
                          Apertus-8B           70.801255  71.200456   
    

In [230]:
# Detailed results on MGSM
df_task = df[("Multilingual", "STEM", "MGSM")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU             \
                                                      de         es   
Open-source  European     EuroLLM-9B           70.266667  71.600000   
                          EuroLLM-22B-Preview  72.666667  75.200000   
                          EuroLLM-22B          76.933333  77.333333   
                          Apertus-8B           59.733333  62.400000   
                          Apertus-70B          74.400000  75.200000   
             Non-European OLMo-3-7B            75.600000  83.200000   
                          OLMo-3.1-32B         88.000000  91.066667   
Open-weights European     Mistral-3.2-24B      90.666667  92.266667   
             Non-European Llama-3.1-8B         74.533333  77.866667   
                          Llama-3.3-70B        92.800000  94.000000   
                          Gemma-3-12B          88.400000  90.666667   
                          Gemma-3-27B          88.933333  91.600000   
                          Qwen-3-14B           90.266667  91.200000   
                          Qwen-3-32B           92.400000  93.066667   
                          Qwen-3-30B-A3B       91.200000  94.000000   

                                                             Non-EU  \
                                                      fr         ja   
Open-source  European     EuroLLM-9B           71.200000  57.600000   
                          EuroLLM-22B-Preview  73.733333  64.000000   
                          EuroLLM-22B          79.066667  67.866667   
                          Apertus-8B           62.000000  49.466667   
                          Apertus-70B          71.200000  68.000000   
             Non-European OLMo-3-7B            82.933333  61.466667   
                          OLMo-3.1-32B         87.333333  81.333333   
Open-weights European     Mistral-3.2-24B      89.333333  83.866667   
             Non-European Llama-3.1-8B         74.400000  60.666667   
                          Llama-3.3-70B        92.133333  88.800000   
                          Gemma-3-12B          83.466667  81.733333   
                          Gemma-3-27B          89.200000  83.600000   
                          Qwen-3-14B           89.466667  86.933333   
                          Qwen-3-32B           90.400000  88.533333   
                          Qwen-3-30B-A3B       88.933333  86.133333   

                                                                     
                                                      ru         zh  
Open-source  European     EuroLLM-9B           71.333333  62.000000  
                          EuroLLM-22B-Preview  75.600000  70.000000  
                          EuroLLM-22B          81.866667  73.333333  
                          Apertus-8B           66.533333  53.200000  
                          Apertus-70B          76.800000  70.800000  
             Non-European OLMo-3-7B            78.666667  78.000000  
                          OLMo-3.1-32B         93.733333  82.800000  
Open-weights European     Mistral-3.2-24B      92.533333  88.800000  
             Non-European Llama-3.1-8B         77.600000  72.933333  
                          Llama-3.3-70B        92.933333  89.200000  
                          Gemma-3-12B          87.333333  84.666667  
                          Gemma-3-27B          90.666667  86.666667  
                          Qwen-3-14B           92.800000  89.600000  
                          Qwen-3-32B           94.533333  91.200000  
                          Qwen-3-30B-A3B       92.933333  90.000000

In [231]:
# Detailed results on FLORES
df_task = df[("Multilingual", "Translation", "FLORES")]
df_task.columns = pd.MultiIndex.from_tuples(
    [
        (
            "EU" if col in EU_LANGS else "Non-EU",
            "en-xx" if col.startswith("en") else "xx-en",
            col,
        )
        for col in df_task.columns
    ]
)
df_task = df_task[
    [
        (col1, col2, col3)
        for col1 in ["EU", "Non-EU"]
        for col2 in ["en-xx", "xx-en"]
        for col3 in df_task[(col1, col2)].columns
    ]
]
df_task

EU             \
                                                   en-xx              
                                                   en-bg      en-cs   
Open-source  European     EuroLLM-9B           91.655567  92.039005   
                          EuroLLM-22B-Preview  91.639162  92.235933   
                          EuroLLM-22B          91.786542  92.326388   
                          Apertus-8B           90.630762  90.921834   
                          Apertus-70B          91.146332  91.498167   
             Non-European OLMo-3-7B            57.260832  49.129471   
                          OLMo-3.1-32B         78.980691  75.726386   
Open-weights European     Mistral-3.2-24B      88.935361  89.441839   
             Non-European Llama-3.1-8B         85.498166  88.158769   
                          Llama-3.3-70B        90.074248  91.152596   
                          Gemma-3-12B          91.332576  91.227186   
                          Gemma-3-27B          91.779802  92.160313   
                          Qwen-3-14B           88.186284  88.856031   
                          Qwen-3-32B           88.450368  88.384870   
                          Qwen-3-30B-A3B       89.540785  89.645756   

                                                                     \
                                                                      
                                                   en-da      en-de   
Open-source  European     EuroLLM-9B           91.668568  88.677733   
                          EuroLLM-22B-Preview  91.853237  89.000295   
                          EuroLLM-22B          91.775105  88.865668   
                          Apertus-8B           90.835675  88.127839   
                          Apertus-70B          90.872055  88.469978   
             Non-European OLMo-3-7B            63.317627  77.193092   
                          OLMo-3.1-32B         83.512576  86.388103   
Open-weights European     Mistral-3.2-24B      90.285858  87.561983   
             Non-European Llama-3.1-8B         88.292587  86.439089   
                          Llama-3.3-70B        91.057703  88.249005   
                          Gemma-3-12B          91.435875  88.282390   
                          Gemma-3-27B          91.714826  88.886226   
                          Qwen-3-14B           87.973382  87.664541   
                          Qwen-3-32B           88.280543  87.892684   
                          Qwen-3-30B-A3B       88.708048  87.864744   

                                                                     \
                                                                      
                                                   en-el      en-es   
Open-source  European     EuroLLM-9B           90.006932  87.244344   
                          EuroLLM-22B-Preview  90.261048  87.411430   
                          EuroLLM-22B          90.089020  87.409248   
                          Apertus-8B           89.210070  86.764748   
                          Apertus-70B          89.531614  86.668994   
             Non-European OLMo-3-7B            47.567455  83.872568   
                          OLMo-3.1-32B         72.086202  86.355554   
Open-weights European     Mistral-3.2-24B      87.784873  85.148146   
             Non-European Llama-3.1-8B         84.320084  85.680889   
                          Llama-3.3-70B        88.055825  86.647306   
                          Gemma-3-12B          89.904380  87.192737   
                          Gemma-3-27B          90.103299  87.304194   
                          Qwen-3-14B           85.474358  86.525680   
                          Qwen-3-32B           85.391123  86.729630   
                          Qwen-3-30B-A3B       86.977552  87.018999   

                                                                     \
                                                                      
                                                   en-et      en-fi   
Open

In [232]:
# Detailed results on WMT24++
df_task = df[("Multilingual", "Translation", "WMT24++")]
df_task.columns = pd.MultiIndex.from_tuples(
    [
        (
            "EU" if col in EU_LANGS else "Non-EU",
            "en-xx" if col.startswith("en") else "xx-en",
            col,
        )
        for col in df_task.columns
    ]
)
df_task = df_task[
    [
        (col1, col2, col3)
        for col1 in ["EU", "Non-EU"]
        for col2 in ["en-xx", "xx-en"]
        for col3 in df_task[(col1, col2)].columns
    ]
]
df_task

EU             \
                                                   en-xx              
                                                   en-bg      en-cs   
Open-source  European     EuroLLM-9B           85.591527  84.978642   
                          EuroLLM-22B-Preview  86.037830  85.825050   
                          EuroLLM-22B          86.068231  85.714973   
                          Apertus-8B           84.140367  82.201322   
                          Apertus-70B          83.643767  83.541309   
             Non-European OLMo-3-7B            51.226646  46.388012   
                          OLMo-3.1-32B         70.784880  66.790132   
Open-weights European     Mistral-3.2-24B      79.440701  79.357687   
             Non-European Llama-3.1-8B         75.604477  78.025881   
                          Llama-3.3-70B        82.776599  83.471007   
                          Gemma-3-12B          85.431178  84.142765   
                          Gemma-3-27B          86.477247  85.578769   
                          Qwen-3-14B           82.264502  80.590881   
                          Qwen-3-32B           82.684586  81.110433   
                          Qwen-3-30B-A3B       83.543879  82.143902   

                                                                     \
                                                                      
                                                   en-da      en-de   
Open-source  European     EuroLLM-9B           85.201540  82.454082   
                          EuroLLM-22B-Preview  85.633880  82.747863   
                          EuroLLM-22B          85.350077  82.545354   
                          Apertus-8B           83.481325  79.955583   
                          Apertus-70B          83.471690  80.681612   
             Non-European OLMo-3-7B            57.031815  67.936391   
                          OLMo-3.1-32B         75.333565  78.970495   
Open-weights European     Mistral-3.2-24B      80.556499  78.395560   
             Non-European Llama-3.1-8B         79.965314  78.455591   
                          Llama-3.3-70B        84.397047  81.778256   
                          Gemma-3-12B          85.414088  81.906418   
                          Gemma-3-27B          86.238478  82.216157   
                          Qwen-3-14B           80.660016  80.990330   
                          Qwen-3-32B           81.663782  81.424470   
                          Qwen-3-30B-A3B       81.597568  81.772537   

                                                                     \
                                                                      
                                                   en-el      en-es   
Open-source  European     EuroLLM-9B           86.007254  83.102897   
                          EuroLLM-22B-Preview  86.547288  83.827353   
                          EuroLLM-22B          86.566466  83.735114   
                          Apertus-8B           84.358122  81.593958   
                          Apertus-70B          84.503872  80.790229   
             Non-European OLMo-3-7B            46.576315  76.810617   
                          OLMo-3.1-32B         66.467675  81.725141   
Open-weights European     Mistral-3.2-24B      80.350826  79.869276   
             Non-European Llama-3.1-8B         78.092087  80.891475   
                          Llama-3.3-70B        84.081621  82.844786   
                          Gemma-3-12B          85.802638  83.215077   
                          Gemma-3-27B          86.950591  84.071492   
                          Qwen-3-14B           80.970362  82.563613   
                          Qwen-3-32B           81.969533  82.926766   
                          Qwen-3-30B-A3B       82.345312  82.811202   

                                                                     \
                                                                      
                                                   en-et      en-fi   
Open

In [233]:
# Detailed results on WMT25
df_task = df[("Multilingual", "Translation", "WMT25")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU             \
                                                   en-cs      en-et   
Open-source  European     EuroLLM-9B           82.194619  80.042507   
                          EuroLLM-22B-Preview        NaN        NaN   
                          EuroLLM-22B          81.658845  80.080010   
                          Apertus-8B           78.105919  78.328798   
                          Apertus-70B          81.930951  81.096183   
             Non-European OLMo-3-7B            43.851382  34.724931   
                          OLMo-3.1-32B         65.444412  47.510256   
Open-weights European     Mistral-3.2-24B      71.419265  64.732177   
             Non-European Llama-3.1-8B         75.969195  63.898256   
                          Llama-3.3-70B        79.686877  76.308593   
                          Gemma-3-12B          85.096990  80.793012   
                          Gemma-3-27B          86.425440  83.801540   
                          Qwen-3-14B           78.574201  67.218038   
                          Qwen-3-32B           81.322084  69.229368   
                          Qwen-3-30B-A3B       82.310552  71.018317   

                                                             Non-EU  \
                                                   cs-de      en-ar   
Open-source  European     EuroLLM-9B           78.846006  71.781821   
                          EuroLLM-22B-Preview        NaN        NaN   
                          EuroLLM-22B          77.976900  70.143095   
                          Apertus-8B           79.133340  70.687028   
                          Apertus-70B          80.043211  73.046088   
             Non-European OLMo-3-7B            55.204252  62.819647   
                          OLMo-3.1-32B         75.051335  71.842783   
Open-weights European     Mistral-3.2-24B      74.531794  65.320448   
             Non-European Llama-3.1-8B         76.741796  66.439842   
                          Llama-3.3-70B        80.457324  67.511482   
                          Gemma-3-12B          81.359308  75.097416   
                          Gemma-3-27B          81.504646  74.800620   
                          Qwen-3-14B           79.253444  72.542379   
                          Qwen-3-32B           81.283500  73.827489   
                          Qwen-3-30B-A3B       81.801022  74.885388   

                                                                     \
                                                   en-ja      en-ko   
Open-source  European     EuroLLM-9B           83.294993  80.483983   
                          EuroLLM-22B-Preview        NaN        NaN   
                          EuroLLM-22B          82.805766  81.421146   
                          Apertus-8B           81.343454  79.644779   
                          Apertus-70B          85.941608  82.774494   
             Non-European OLMo-3-7B            73.163966  65.659513   
                          OLMo-3.1-32B         84.022857  82.624173   
Open-weights European     Mistral-3.2-24B      82.199767  75.449232   
             Non-European Llama-3.1-8B         78.667851  73.555670   
                          Llama-3.3-70B        84.015005  81.090569   
                          Gemma-3-12B          88.091159  87.276960   
                          Gemma-3-27B          88.274945  87.290364   
                          Qwen-3-14B           87.223082  85.495633   
                          Qwen-3-32B           88.233138  86.345378   
                          Qwen-3-30B-A3B       88.139460  86.678342   

                                                                     \
                                                   en-ru      en-uk   
Open-source  European     EuroLLM-9B           80.219896  81.016137   
                          EuroLLM-22B-Preview        NaN        NaN   
                          EuroLLM-22B          78.928951  79.531989   
                          Apertus-8B           79.803152  79.991731   
    

## EuroMOE-2.6B-A0.6B

In [234]:
# Define models
MODELS = [
    "EuroLLM-1.7B",
    "EuroMoE-2.6B-A0.6B",
]

In [235]:
# Format results as dataframe
df = pd.DataFrame()

for task_group_1 in GROUPED_TASKS:
    is_multilingual = task_group_1 == "Multilingual"

    for task_group_2 in GROUPED_TASKS[task_group_1]:
        is_mt = task_group_2 == "Translation"

        for task in GROUPED_TASKS[task_group_1][task_group_2]:
            if is_multilingual:
                if is_mt:
                    lps = results[task]
                    lps = (
                        sorted([lp for lp in lps if lp.startswith("en")])
                        + sorted([lp for lp in lps if lp.endswith("en")])
                        + sorted(
                            [
                                lp
                                for lp in lps
                                if not (lp.startswith("en") or lp.endswith("en"))
                            ]
                        )
                    )
                    for lp in lps:
                        for model in MODELS:
                            try:
                                df.loc[
                                    model, f"{task_group_1}_{task_group_2}_{task}_{lp}"
                                ] = np.mean(
                                    [
                                        results[task][lp][model][judge]
                                        for judge in MT_JUDGES
                                    ]
                                )
                            except:
                                df.loc[
                                    model, f"{task_group_1}_{task_group_2}_{task}_{lp}"
                                ] = None
                else:
                    langs = sorted(results[task].keys())
                    for lang in langs:
                        for model in MODELS:
                            try:
                                df.loc[
                                    model,
                                    f"{task_group_1}_{task_group_2}_{task}_{lang}",
                                ] = np.mean(
                                    [
                                        results[task][lang][model][judge]
                                        for judge in JUDGES
                                    ]
                                )
                            except:
                                df.loc[
                                    model,
                                    f"{task_group_1}_{task_group_2}_{task}_{lang}",
                                ] = None
            else:
                for model in MODELS:
                    try:
                        df.loc[model, f"{task_group_1}_{task_group_2}_{task}_en"] = (
                            np.nanmean(
                                [results[task]["en"][model][judge] for judge in JUDGES]
                            )
                        )
                    except:
                        df.loc[model, f"{task_group_1}_{task_group_2}_{task}_en"] = None

df.columns = pd.MultiIndex.from_tuples([col.split("_") for col in df.columns])

In [236]:
# Aggregate results
df.loc["EuroLLM-1.7B", ("Multilingual", "Translation", "WMT25")] = None
df_agg = df.groupby(level=[0, 1, 2], axis=1).mean()
df_agg = df_agg[
    [
        (col1, col2, col3)
        for col1 in GROUPED_TASKS
        for col2 in GROUPED_TASKS[col1]
        for col3 in GROUPED_TASKS[col1][col2]
    ]
]

df_en = df_agg["English"]

df_xx = df_agg["Multilingual"]

df_eu = df["Multilingual"].copy()
cols = df_eu.columns
for col in cols:
    if col[-1] not in EU_LANGS:
        df_eu = df_eu.drop(col, axis=1)
df_eu = df_eu.groupby(level=[0, 1], axis=1).mean()
df_eu = df_eu[
    [
        (group, task)
        for group, tasks in GROUPED_TASKS["Multilingual"].items()
        for task in tasks
        if (group, task) in df_eu.columns
    ]
]

df_non_eu = df["Multilingual"].copy()
cols = df_non_eu.columns
for col in cols:
    if col[-1] not in NON_EU_LANGS:
        df_non_eu = df_non_eu.drop(col, axis=1)
df_non_eu = df_non_eu.groupby(level=[0, 1], axis=1).mean()
df_non_eu = df_non_eu[
    [
        (group, task)
        for group, tasks in GROUPED_TASKS["Multilingual"].items()
        for task in tasks
        if (group, task) in df_non_eu.columns
    ]
]

In [237]:
# English results
df_en

Instruction-following    General                        \
                                  IFEval  Hellaswag       MMLU   MMLU-Pro   
EuroLLM-1.7B                    9.796673  18.721370  26.121635  10.123005   
EuroMoE-2.6B-A0.6B             24.707332  28.145124  36.860846  18.941157   

                                    STEM                                  \
                          BBH      ARC-C GPQA $\blacklozenge$      GSM8K   
EuroLLM-1.7B        28.244509  27.929465            12.289562  20.368966   
EuroMoE-2.6B-A0.6B  23.688118  44.112628            16.835017  24.488249   

                                          
                     MATH-500  HumanEval  
EuroLLM-1.7B         3.933333  25.203252  
EuroMoE-2.6B-A0.6B  10.666667  22.764228

In [238]:
# Multilingual results
df_xx

General                             STEM             \
                   M-Hellaswag      MMMLU  MMLU-ProX    M-ARC-C       MGSM   
EuroLLM-1.7B         23.664985  22.873823  10.840936  21.820491   6.333333   
EuroMoE-2.6B-A0.6B   19.206159  31.146607  14.045372  35.438961  14.133333   

                   Translation                        
                        FLORES    WMT24++      WMT25  
EuroLLM-1.7B         86.808637  79.872413        NaN  
EuroMoE-2.6B-A0.6B   86.814901  79.802740  71.868167

In [239]:
# Results on European languages
df_eu

General                             STEM             \
                   M-Hellaswag      MMMLU  MMLU-ProX    M-ARC-C       MGSM   
EuroLLM-1.7B         23.404116  23.250895  11.414560  22.441847   8.177778   
EuroMoE-2.6B-A0.6B   19.464096  32.061898  14.531929  36.379772  14.888889   

                   Translation                        
                        FLORES    WMT24++      WMT25  
EuroLLM-1.7B         86.886003  80.248814        NaN  
EuroMoE-2.6B-A0.6B   86.878234  80.179617  72.429484

In [240]:
# Results on non-European languages
df_non_eu

General                             STEM             \
                   M-Hellaswag      MMMLU  MMLU-ProX    M-ARC-C       MGSM   
EuroLLM-1.7B         24.291069  22.119680  10.267313  20.577780   4.488889   
EuroMoE-2.6B-A0.6B   18.587111  29.316025  13.558814  33.557338  13.377778   

                   Translation                        
                        FLORES    WMT24++      WMT25  
EuroLLM-1.7B         86.619522  79.081969        NaN  
EuroMoE-2.6B-A0.6B   86.660087  79.011299  71.657673

In [241]:
# Detailed results on M-Hellaswag
df_task = df[("Multilingual", "General", "M-Hellaswag")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU                                              \
                           da         de         es         fr         hr   
EuroLLM-1.7B        24.707147  26.590521  24.489723  17.216392  27.117869   
EuroMoE-2.6B-A0.6B  19.788644  24.081981  21.481402  21.207254  19.075775   

                                                                           \
                           hu         it         nl         pt         ro   
EuroLLM-1.7B        18.247615  21.262555  25.285123  24.036551  21.856860   
EuroMoE-2.6B-A0.6B  13.141061  21.820951  19.107753  20.522267  16.617992   

                                             Non-EU                        \
                           sk         sv         ar         ca         hi   
EuroLLM-1.7B        25.373397  24.665643  26.540250  22.509319  24.107901   
EuroMoE-2.6B-A0.6B  16.078018  20.646057  20.546353  17.862700  19.201359   

                                          
                           ru         uk  
EuroLLM-1.7B        23.367846  24.930032  
EuroMoE-2.6B-A0.6B  17.817084  17.508060

In [242]:
# Detailed results on MMMLU
df_task = df[("Multilingual", "General", "MMMLU")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU                                              \
                           da         de         es         fr         hr   
EuroLLM-1.7B        23.832601  23.153316  24.961252  25.656303  19.741913   
EuroMoE-2.6B-A0.6B  31.576556  33.861316  35.383231  35.013877  29.533009   

                                                                           \
                           hu         it         nl         pt         ro   
EuroLLM-1.7B        19.740479  25.184458  25.347196  24.056840  22.162638   
EuroMoE-2.6B-A0.6B  25.081900  34.320465  32.746452  32.167517  32.004028   

                                             Non-EU                        \
                           sk         sv         ar         ca         hi   
EuroLLM-1.7B        22.365130  22.808609  19.771040  24.427384  20.522098   
EuroMoE-2.6B-A0.6B  31.450008  31.604416  27.792389  31.164488  26.565892   

                                                    
                           ru         uk        zh  
EuroLLM-1.7B        23.548858  21.486302  22.96240  
EuroMoE-2.6B-A0.6B  29.842905  29.493635  31.03684

In [243]:
# Detailed results on MMLU-ProX
df_task = df[("Multilingual", "General", "MMLU-ProX")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU                                              \
                           cs         de         es         fr         hu   
EuroLLM-1.7B        10.842759  11.769708  11.939791  12.112708   9.745727   
EuroMoE-2.6B-A0.6B  13.955268  14.760325  16.075630  15.534201  11.902940   

                                             Non-EU                        \
                           it         pt         ar         hi         ja   
EuroLLM-1.7B        11.585452  11.905774  11.092213   9.765570   9.887462   
EuroMoE-2.6B-A0.6B  15.097656  14.397483  13.368484  13.283442  12.401848   

                                                                
                           ko         ru         uk         zh  
EuroLLM-1.7B         9.008703  10.984494  10.502594  10.630156  
EuroMoE-2.6B-A0.6B  12.920600  14.411656  14.173541  14.352127

In [244]:
# Detailed results on M-ARC-C
df_task = df[("Multilingual", "STEM", "M-ARC-C")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU                                              \
                           da         de         es         fr         hr   
EuroLLM-1.7B        21.908026  20.958084  25.584046  24.037639  17.992586   
EuroMoE-2.6B-A0.6B  32.790631  37.838608  39.059829  42.372398  30.367836   

                                                                           \
                           hu         it         nl         pt         ro   
EuroLLM-1.7B        19.463470  24.579413  23.923581  23.732194  21.993716   
EuroMoE-2.6B-A0.6B  26.712329  40.490448  40.148275  36.495726  35.075693   

                                             Non-EU                        \
                           sk         sv         ar         ca         hi   
EuroLLM-1.7B        22.725977  22.403433  16.566866  23.098914  18.122146   
EuroMoE-2.6B-A0.6B  38.066724  37.138770  31.593955  37.049743  28.852740   

                                                     
                           ru         uk         zh  
EuroLLM-1.7B        22.954092  21.186199  21.538462  
EuroMoE-2.6B-A0.6B  34.816082  33.532934  35.498575

In [245]:
# Detailed results on MGSM
df_task = df[("Multilingual", "STEM", "MGSM")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU                          Non-EU             \
                           de         es         fr        ja         ru   
EuroLLM-1.7B         6.666667   6.800000  11.066667  2.666667   8.266667   
EuroMoE-2.6B-A0.6B  15.333333  15.066667  14.266667  9.466667  16.400000   

                               
                           zh  
EuroLLM-1.7B         2.533333  
EuroMoE-2.6B-A0.6B  14.266667

In [246]:
# Detailed results on FLORES
df_task = df[("Multilingual", "Translation", "FLORES")]
df_task.columns = pd.MultiIndex.from_tuples(
    [
        (
            "EU" if col in EU_LANGS else "Non-EU",
            "en-xx" if col.startswith("en") else "xx-en",
            col,
        )
        for col in df_task.columns
    ]
)
df_task = df_task[
    [
        (col1, col2, col3)
        for col1 in ["EU", "Non-EU"]
        for col2 in ["en-xx", "xx-en"]
        for col3 in df_task[(col1, col2)].columns
    ]
]
df_task

EU                                              \
                        en-xx                                               
                        en-bg      en-cs      en-da      en-de      en-el   
EuroLLM-1.7B        89.483317  89.234176  89.563137  86.969147  87.808113   
EuroMoE-2.6B-A0.6B  89.788179  89.935334  89.964416  87.375208  88.451721   

                                                                           \
                                                                            
                        en-es      en-et      en-fi      en-fr      en-ga   
EuroLLM-1.7B        86.334107  88.686035  89.413218  87.637544  74.493990   
EuroMoE-2.6B-A0.6B  86.242436  89.183115  89.596211  87.974279  73.123119   

                    ...     Non-EU                                   \
                    ...      en-xx      xx-en                         
                    ...      en-zh      ca-en      gl-en      hi-en   
EuroLLM-1.7B        ...  86.649845  87.673183  87.812508  88.260172   
EuroMoE-2.6B-A0.6B  ...  86.830472  86.929305  87.794742  87.700783   

                                                                           \
                                                                            
                        ja-en      ko-en      ru-en      tr-en      uk-en   
EuroLLM-1.7B        86.803939  86.779674  85.991577  88.048047  86.275213   
EuroMoE-2.6B-A0.6B  86.735158  86.538679  86.067705  86.610874  86.472472   

                               
                               
                        zh-en  
EuroLLM-1.7B        86.258008  
EuroMoE-2.6B-A0.6B  86.084734  

[2 rows x 62 columns]

In [247]:
# Detailed results on WMT24++
df_task = df[("Multilingual", "Translation", "WMT24++")]
df_task.columns = pd.MultiIndex.from_tuples(
    [
        (
            "EU" if col in EU_LANGS else "Non-EU",
            "en-xx" if col.startswith("en") else "xx-en",
            col,
        )
        for col in df_task.columns
    ]
)
df_task = df_task[
    [
        (col1, col2, col3)
        for col1 in ["EU", "Non-EU"]
        for col2 in ["en-xx", "xx-en"]
        for col3 in df_task[(col1, col2)].columns
    ]
]
df_task

EU                                              \
                        en-xx                                               
                        en-bg      en-cs      en-da      en-de      en-el   
EuroLLM-1.7B        81.389470  79.786823  81.316282  78.985458  81.969539   
EuroMoE-2.6B-A0.6B  81.888889  81.265679  81.612998  79.285217  82.641665   

                                                                           \
                                                                            
                        en-es      en-et      en-fi      en-fr      en-hr   
EuroLLM-1.7B        80.364283  81.866885  62.743511  78.094604  74.981846   
EuroMoE-2.6B-A0.6B  80.786192  81.759009  62.606865  78.740780  76.692365   

                    ...     Non-EU                                   \
                    ...      xx-en                                    
                    ...      ar-en      ca-en      hi-en      ja-en   
EuroLLM-1.7B        ...  75.134104  80.173018  82.510585  80.145005   
EuroMoE-2.6B-A0.6B  ...  73.952596  78.568085  81.851621  79.892810   

                                                                           \
                                                                            
                        ko-en      no-en      ru-en      tr-en      uk-en   
EuroLLM-1.7B        81.191350  84.197877  78.013068  82.682019  81.074219   
EuroMoE-2.6B-A0.6B  80.937376  83.796445  77.788847  80.610597  80.684726   

                               
                               
                        zh-en  
EuroLLM-1.7B        80.906788  
EuroMoE-2.6B-A0.6B  80.436264  

[2 rows x 62 columns]

In [248]:
# Detailed results on WMT25
df_task = df[("Multilingual", "Translation", "WMT25")]
df_task.columns = pd.MultiIndex.from_tuples(
    [("EU" if col in EU_LANGS else "Non-EU", col) for col in df_task.columns]
)
df_task = pd.concat([df_task[["EU"]], df_task[["Non-EU"]]], axis=1)
df_task

EU                           Non-EU             \
                        en-cs      en-et      cs-de      en-ar      en-ja   
EuroLLM-1.7B             None       None        NaN       None       None   
EuroMoE-2.6B-A0.6B  72.637978  72.861334  71.789141  65.236877  73.809928   

                                                                          \
                        en-ko      en-ru     en-uk      en-zh      cs-uk   
EuroLLM-1.7B             None       None      None       None        NaN   
EuroMoE-2.6B-A0.6B  69.571231  74.436732  73.18348  73.966982  78.454958   

                               
                        ja-zh  
EuroLLM-1.7B             None  
EuroMoE-2.6B-A0.6B  64.601192